# Harry Potter RAG System: Retrieval, Generation, and Evaluation

## 1. Document Preparation

In [2]:
!pip install pymupdf qdrant-client langchain langchain-groq langchain-google-genai

In [3]:
import pymupdf

def pdf_to_markdown(pdf_path, markdown_path):
    """Extract the text from a PDF and save it as a Markdown file."""

    markdown = []

    with pymupdf.open(pdf_path) as pdf:

        total_pages = len(pdf)

        for page_number, page in enumerate(pdf, start=1):
            markdown.append(f"## Page {page_number}\n\n")
            markdown.append(page.get_text())
            markdown.append("\n\n")

    with open(markdown_path, "w", encoding="utf-8") as file:
        file.write("".join(markdown))

    print(f"Total Pages: {total_pages}")
    print(f"Markdown file created: {markdown_path}")


pdf_to_markdown(
    "/kaggle/input/datasets/nadotarek133/harry-potter-rag/harrypotter.pdf",
    "output.md",
)

Total Pages: 3623
Markdown file created: output.md


## 2. Page-Level Chunking

In [4]:
from pathlib import Path
import re

INPUT_FILE = Path("output.md")
OUTPUT_FOLDER = Path("dataset")

BOOK_RANGES = [
    ("Harry Potter and the Sorcerer Stone", 12, 274),
    ("Harry Potter and the Chamber of Secrets", 282, 565),
    ("Harry Potter and the Prisoner of Azkaban", 573, 939),
    ("Harry Potter and the Goblet of Fire", 949, 1560),
    ("Harry Potter and the Order of the Phoenix", 1570, 2406),
    ("Harry Potter and the Half-Blood Prince", 2409, 2964),
    ("Harry Potter and the Deathly Hallows", 2974, 3622),
]


def get_book_name(page_number):
    """Helper Function"""
    """Return the book name for a given page number."""

    for book_name, first_page, last_page in BOOK_RANGES:
        if first_page <= page_number <= last_page:
            return book_name

    return None

def clean_text(text):
    """Helper Function"""
    """Clean the text by removing extra whitespace and unwanted characters."""

    text = re.sub(r'\s+', ' ', text)
    text = text.replace(r'[\r\n]+', '\n').strip()

    return text


def split_pages():
    """Split the Markdown file into separate files for each page."""

    text = INPUT_FILE.read_text(encoding="utf-8")

    pages = re.split(r"^##\s*Page\s+(\d+)\s*$", text, flags=re.MULTILINE)

    OUTPUT_FOLDER.mkdir(exist_ok=True)

    for i in range(1, len(pages), 2):

        page_number = int(pages[i])
        page_content = clean_text(pages[i + 1])
        book_name = get_book_name(page_number)

        if book_name:

            markdown = (
                f"# {book_name}\n\n"
                f"## Page {page_number}\n\n"
                f"{page_content}"
            )

            file_name = f"{book_name} - Page {page_number}.md"
            (OUTPUT_FOLDER / file_name).write_text(markdown, encoding="utf-8")



split_pages()


In [5]:
from collections import Counter

books_counter = Counter()

for file in OUTPUT_FOLDER.glob("*.md"):
    book_name = file.name.split(" - Page ")[0]
    books_counter[book_name] += 1

books_counter

Counter({'Harry Potter and the Order of the Phoenix': 837,
         'Harry Potter and the Sorcerer Stone': 263,
         'Harry Potter and the Chamber of Secrets': 284,
         'Harry Potter and the Half-Blood Prince': 556,
         'Harry Potter and the Prisoner of Azkaban': 367,
         'Harry Potter and the Goblet of Fire': 612,
         'Harry Potter and the Deathly Hallows': 649})

In [6]:
import plotly.express as px
import pandas as pd

books_df = pd.DataFrame(
    books_counter.items(),
    columns=["Book", "Pages"]
)

fig = px.bar(
    books_df,
    x="Book",
    y="Pages",
    title="Pages per Book"
)

fig.show()

## 3. Embedding Generation


In [7]:
import torch
from sentence_transformers import SentenceTransformer
from pathlib import Path
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

DATASET_FOLDER = Path("dataset")
MODEL_NAME = user_secrets.get_secret("EMBEDDING_MODEL")

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


def read_page(file_path):
    lines = file_path.read_text(encoding="utf-8").splitlines()

    book_name = lines[0].replace("# ", "")
    page_number = int(lines[2].replace("## Page ", ""))
    content = " ".join(lines[3:]).strip()

    return {
        "book_name": book_name,
        "page_number": page_number,
        "content": content,
    }



files = sorted(DATASET_FOLDER.glob("*.md"))
pages = [read_page(file) for file in files]
texts = [f"passage: {page['content']}" for page in pages]

model = SentenceTransformer(MODEL_NAME , device=device)


embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
).tolist()

Using device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

Batches:   0%|          | 0/112 [00:00<?, ?it/s]

## 4. Vector Database Indexing

In [8]:
from qdrant_client import QdrantClient, models
import os


QDRANT_URL = user_secrets.get_secret("QDRANT_URL")
QDRANT_API_KEY = user_secrets.get_secret("QDRANT_API_KEY")
QDRANT_COLLECTION = user_secrets.get_secret("QDRANT_COLLECTION")


client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
)

collection_name = QDRANT_COLLECTION
vector_size = len(embeddings[0])

if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size=vector_size,
            distance=models.Distance.COSINE,
        ),
    )


points = [
    models.PointStruct(
        id=index,
        vector=embedding,
        payload=page,
    )
    for index, (page, embedding) in enumerate(zip(pages, embeddings))
]

batch_size = 100

for start in range(0, len(points), batch_size):
    batch = points[start:start + batch_size]
    client.upsert(
        collection_name=collection_name,
        points=batch,
    )

print(f"Uploaded {len(points)} pages to Qdrant.")

Uploaded 3568 pages to Qdrant.


## 5. Query Routing

In [9]:
import os
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

query = input("Ask a question: " )

router_llm = ChatGroq(
    model=user_secrets.get_secret("GROQ_MODEL"),
    api_key=user_secrets.get_secret("GROQ_API_KEY"),
    temperature=0,
)

SYSTEM_PROMPT = SYSTEM_PROMPT = """
You classify messages for a Harry Potter book search system.

Return exactly one label and nothing else:

retrieve - questions about the books, characters, places, spells, creatures, or events.

chitchat - greetings, thanks, or casual conversation, and in this case you can answer the question.

off-topic - anything unrelated to the Harry Potter books, and tell the user that you are only answering questions about the Harry Potter books.

Return only one label:
retrieve
chitchat
off-topic
"""

router_messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=query),
]


route = router_llm.invoke(router_messages).content.strip().lower()
route = route.splitlines()[0].strip(" `.,:")

if route not in {"retrieve", "chitchat", "off-topic"}:
    route = "off-topic"

print("Route:", route)


Ask a question:  Who is the headmaster of Hogwarts?


Route: retrieve


## 6. Semantic Retrieval

In [10]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

try:
    top_k = int(user_secrets.get_secret("TOP_K"))
except:
    top_k = 3

if route == "retrieve":
    query_vector = model.encode(
        [f"query: {query}"],
        normalize_embeddings=True,
    )[0].tolist()

    results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
    ).points


    context = "Retrieved Pages:\n\n"
    for result in results:
        page = result.payload
        context += (
            f"Book: {page['book_name']}\n"
            f"Page: {page['page_number']}\n"
            f"Content: {page['content']}\n\n"
        )

        print("Score:", result.score)
        print("Book:", page["book_name"])
        print("Page:", page["page_number"])
        print("Content:", page["content"])
        print("-" * 80)

else:
    print("No database search needed.")

Score: 0.8253851
Book: Harry Potter and the Order of the Phoenix
Page: 1768
Content: But Professor McGonagall, who was waiting to read out the list of first years’ names, was giving the whispering students the sort of look that scorches. Nearly Headless Nick placed a see-through finger to his lips and sat primly upright again as the muttering came to an abrupt end. With a last frowning look that swept the four House tables, Professor McGonagall lowered her eyes to her long piece of parchment and called out, “Abercrombie, Euan.” The terrified-looking boy Harry had noticed earlier stumbled forward and put the hat on his head; it was only prevented from falling right down to his shoulders by his very prominent ears. The hat considered for a moment, then the rip near the brim opened again and shouted, “GRYFFINDOR!” Harry clapped loudly with the rest of Gryffindor House as Euan Abercrombie staggered to their table and sat down, looking as though he would like very much to sink through the f

## 7. Keyword Search

In [11]:
if route == "retrieve":
    keyword_query = "Hogwarts"
    top_k = 3

    keywords = keyword_query.lower().split()
    keyword_results = []

    for page in pages:
        content = page["content"].lower()
        score = sum(content.count(keyword) for keyword in keywords)

        if score > 0:
            keyword_results.append({
                "score": score,
                "book_name": page["book_name"],
                "page_number": page["page_number"],
                "content": page["content"],
            })

    keyword_results.sort(key=lambda result: result["score"], reverse=True)

    for result in keyword_results[:top_k]:
        print("Keyword score:", result["score"])
        print("Book:", result["book_name"])
        print("Page:", result["page_number"])
        print("Content:", result["content"])
        print("-" * 80)


Keyword score: 5
Book: Harry Potter and the Order of the Phoenix
Page: 1865
Content: “This is not the first time in recent weeks Fudge has used new laws to effect improvements at the Wizarding school. As recently as August 30th Educational Decree Twenty-two was passed, to ensure that, in the event of the current headmaster being unable to provide a candidate for a teaching post, the Ministry should select an appropriate person. “‘That’s how Dolores Umbridge came to be appointed to the teaching staff at Hogwarts,’ said Weasley last night. ‘Dumbledore couldn’t find anyone, so the Minister put in Umbridge and of course, she’s been an immediate success —’” “She’s been a WHAT?” said Harry loudly. “Wait, there’s more,” said Hermione grimly. “‘— an immediate success, totally revolutionizing the teaching of Defense Against the Dark Arts and providing the Minister with on-the-ground feedback about what’s really happening at Hogwarts.’ “It is this last function that the Ministry has now formaliz

## 8. Answer Generation

In [12]:
if route == "retrieve":
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain_core.messages import SystemMessage, HumanMessage

    gemini_llm = ChatGoogleGenerativeAI(
        model=user_secrets.get_secret("GEMINI_MODEL"),
        api_key=user_secrets.get_secret("GEMINI_API_KEY"),
        temperature=0,
    )

    messages = [
    SystemMessage(
        content="""
You are a Harry Potter books assistant.

Answer ONLY using the provided context.

Rules:
- Use only the retrieved pages.
- Do not use outside knowledge.
- Do not make assumptions.
- Do not add facts that are not present in the context.
- If the answer cannot be found in the context, reply exactly:
I do not know.
- Cite the page number at the end of the answer using this format:
[Page X]
- Keep the answer concise.
"""
    ),
    HumanMessage(
        content=f"Context:\n{context}\n\nQuestion:\n{query}"
    ),
]
    response = gemini_llm.invoke(messages)
    print("\nAnswer:")
    print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.



Answer:
Professor Dumbledore is the headmaster. [Page 1768]


## 9. Retrieval Evaluation: Precision and Recall

In [13]:
evaluation_cases = [
    {
        "query": "Who rescued Harry from his locked bedroom using a flying car?",
        "relevant_pages": {301, 302},
    },
    {
        "query": "What loophole did Mr Weasley write into the law about enchanting a car?",
        "relevant_pages": {314},
    },
]

top_k = 3
precision_scores = []
recall_scores = []
f1_scores = []
retrieved_for_evaluation = []

for case in evaluation_cases:
    query_vector = model.encode(
        [f"query: {case['query']}"],
        normalize_embeddings=True,
    )[0].tolist()

    search_results = client.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
        with_payload=True,
    ).points

    retrieved_pages = {result.payload["page_number"] for result in search_results}
    relevant_pages = case["relevant_pages"]
    relevant_retrieved = retrieved_pages & relevant_pages

    precision = len(relevant_retrieved) / len(retrieved_pages) if retrieved_pages else 0
    recall = len(relevant_retrieved) / len(relevant_pages)
    f1 = (
    2 * precision * recall / (precision + recall)
    if (precision + recall) > 0
    else 0
    )

    precision_scores.append(precision)
    recall_scores.append(recall)
    f1_scores.append(f1)
    retrieved_for_evaluation.append((case, search_results))

    print(case["query"])
    print("Expected pages:", relevant_pages)
    print("Retrieved pages:", retrieved_pages)
    print(f"Precision: {precision:.2f}")
    print(f"Recall:    {recall:.2f}")
    print(f"F1 Score:  {f1:.2f}")
    print("-" * 60)

print(f"Average precision: {sum(precision_scores) / len(precision_scores):.2f}")
print(f"Average recall:    {sum(recall_scores) / len(recall_scores):.2f}")
print(f"Average F1 Score:  {sum(f1_scores) / len(f1_scores):.2f}")

Who rescued Harry from his locked bedroom using a flying car?
Expected pages: {301, 302}
Retrieved pages: {303, 302, 967}
Precision: 0.33
Recall:    0.50
F1 Score:  0.40
------------------------------------------------------------
What loophole did Mr Weasley write into the law about enchanting a car?
Expected pages: {314}
Retrieved pages: {2056, 314, 467}
Precision: 0.33
Recall:    1.00
F1 Score:  0.50
------------------------------------------------------------
Average precision: 0.33
Average recall:    0.75
Average F1 Score:  0.45


In [14]:
import pandas as pd
import plotly.express as px

metrics_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1"],
    "Score": [
        sum(precision_scores) / len(precision_scores),
        sum(recall_scores) / len(recall_scores),
        sum(f1_scores) / len(f1_scores),
    ]
})

fig = px.bar(
    metrics_df,
    x="Metric",
    y="Score",
    title="Retrieval Evaluation Metrics"
)

fig.show()

## 10. LLM-as-a-Judge Evaluation

In [15]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage

generator_llm = ChatGoogleGenerativeAI(
    model=user_secrets.get_secret("GEMINI_MODEL"),
    api_key=user_secrets.get_secret("GEMINI_API_KEY"),
)

judge_llm = ChatGoogleGenerativeAI(
    model=user_secrets.get_secret("GEMINI_JUDGE"),
    api_key=user_secrets.get_secret("GEMINI_API_KEY"),
)

scores = []

for case, search_results in retrieved_for_evaluation:

    context = "\n\n".join(
        f"Page {result.payload['page_number']}: {result.payload['content']}"
        for result in search_results
    )

    answer = generator_llm.invoke(
        [
            SystemMessage(
                content="""
You are a Harry Potter books assistant.

Answer ONLY using the provided context.

Rules:
- Use only the retrieved pages.
- Do not use outside knowledge.
- Do not make assumptions.
- If the answer is not found in the context, say:
  I do not know.
- Keep the answer concise.
"""
            ),
            HumanMessage(
                content=f"Context:\n{context}\n\nQuestion:\n{case['query']}"
            ),
        ]
    ).text

    judge = judge_llm.invoke(
        [
            SystemMessage(
                content="""
You are an evaluator for a question-answering system.

Evaluate the answer using ONLY the provided context.

Check:
1. Correctness
2. Grounding
3. Completeness

Return exactly in this format:

Score: X/5
Grounded: yes or no
Reason: one short sentence
"""
            ),
            
       HumanMessage(content=f"Context:\n{context}\n\nQuestion:\n{case['query']}\n\nAnswer:\n{answer}"),
    ]).text

    for line in judge.splitlines():
        if line.startswith("Score:"):
            try:
                score = int(
                    line.split(":")[1]
                    .split("/")[0]
                    .strip()
                )
                scores.append(score)
            except:
                pass

    print("Question:", case["query"])
    print("Answer:", answer)
    print("Judge:", judge)
    print("-" * 60)

if scores:
    print(
        f"Average Judge Score: {sum(scores) / len(scores):.2f}/5"
    )

Question: Who rescued Harry from his locked bedroom using a flying car?
Answer: Fred, George, and Ron rescued Harry.
Judge: Score: 5/5
Grounded: yes
Reason: The text explicitly shows Ron, Fred, and George arriving in a flying car to rescue Harry.
------------------------------------------------------------
Question: What loophole did Mr Weasley write into the law about enchanting a car?
Answer: As long as the wizard wasn't intending to fly the car, the fact that the car could fly wouldn't be illegal.
Judge: Score: 5/5
Grounded: yes
Reason: The answer accurately reflects the loophole mentioned in the text.
------------------------------------------------------------
Average Judge Score: 5.00/5
